# WeldSight: SWRD segmentation baseline on a single GPU (Colab T4)

SWRD (Zhao et al., J. Nondestruct. Eval. 2025): ~3,675 seam-weld radiographs, 16-bit, polygon labels for 6 defect classes.
The archive `SWXD_Data.zip` is **~116 GB**, more than Colab's disk and a free Google Drive. We therefore never download it:
`weldsight-remote-subset` reads the zip's table of contents and only the selected files over HTTP Range requests,
and stores images as 8-bit PNG.

Licence is not stated: research use only, cite the paper, do **not** redistribute the images (keep `data/` out of git).

Runtime -> Change runtime type -> T4 GPU. To start over: Runtime -> Disconnect and delete runtime.

In [ ]:
%cd /content
!rm -rf WeldSight && git clone https://github.com/00monaliza/WeldSight.git
%cd /content/WeldSight
!pip -q install uv && uv sync --extra dev

## 1. Look inside the 116 GB archive without downloading it
Reads only the zip's central directory (a few MB). Nothing is written to your Google Drive, so no free Drive space is needed.
We read the file through the Drive API **as you** (sign-in popup), which avoids the anonymous quota ("Too many users have viewed or downloaded this file"). The token lives ~1 h: if a later cell fails with 401, re-run this cell and then the failed one (extraction resumes where it stopped).

In [ ]:
from google.colab import auth
auth.authenticate_user()
import os, google.auth, google.auth.transport.requests
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
creds.refresh(google.auth.transport.requests.Request())
os.environ['GDRIVE_TOKEN'] = creds.token
URL = 'https://www.googleapis.com/drive/v3/files/1Hc9_de5YAdXg46F-GfjcRKMEBRF9XRQi?alt=media&supportsAllDrives=true'
!uv run weldsight-remote-subset --url "$URL" --list

## 1b. Extract all annotations + a seeded subset of images
Start with ~600 images (a few GB after 8-bit conversion); the list is saved to `data/raw/swrd/_subset.txt`
and the command is resumable. Use `--include REGEX` to restrict to a folder (e.g. only standard seam welds).

In [ ]:
!uv run weldsight-remote-subset --url "$URL" --max-images 600 --seed 0
!du -sh data/raw/swrd; find data/raw/swrd -type f | sed 's/.*\.//' | sort | uniq -c

## 2. Inspect labels before preparing
Every label printed under `WARNING: labels not in class_names` must be mapped in `configs/data/swrd.yaml -> source_opts.label_map`.
If several images belong to one weld (e.g. `xxx_1.tif`, `xxx_2.tif`), set `source_opts.group_regex` so they share a split.

In [ ]:
!uv run weldsight-prepare --config configs/experiment/swrd_baseline.yaml --inspect

In [ ]:
!uv run weldsight-prepare --config configs/experiment/swrd_baseline.yaml
!cat data/processed/swrd_v1/dataset_card.json | head -60
!uv run pytest -q

## 3. Train U-Net (ResNet-34 / ImageNet, Dice + Focal) and evaluate on the real test split

In [ ]:
!uv run weldsight-train --config configs/experiment/swrd_baseline.yaml train.num_workers=2

In [ ]:
import glob
from IPython.display import Image, Markdown, display
run = sorted(glob.glob('runs/swrd_baseline_*'))[-1]
!uv run weldsight-eval --run {run} --split val --uncertainty predictive_entropy
display(Markdown(open(f'{run}/results_test.md').read()))
display(Image(f'{run}/referral_test.png'))

Download the whole `runs/<run>` folder (config, provenance with git commit + data version + seed, history, metrics, figures) and send it back: it is enough to reproduce and report the result.

In [ ]:
!rm -f {run}/best.pt.zip; zip -qr /content/swrd_run.zip {run} -x '*.pt'
from google.colab import files; files.download('/content/swrd_run.zip')